# 2.2 Bloom Error Analysis

**Tác giả:** Đăng (TV1)  
**Mục tiêu:**
- Confusion matrix 7×7 đầy đủ
- Per-class F1 + accuracy±1
- Gộp 3 nhóm: {0,1,2} / {3,4} / {5,6}
- Phân tích 20 câu sai phân loại
- Ảnh hưởng OCR quality lên kết quả

In [ ]:
import json
import pickle
import sys
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score, accuracy_score
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
import scipy.sparse as sp

ROOT = Path('..').resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.syllabus_core.visualization.visualize import plot_confusion_matrix

import yaml
with open(ROOT / 'config' / 'params.yaml') as f:
    CFG = yaml.safe_load(f)

RANDOM_SEED = CFG.get('random_seed', 42)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

FIG_DIR = ROOT / 'reports' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
METRICS_PATH = ROOT / 'results' / 'metrics.json'
BLOOM_NAMES = ['Ngoài miền','Nhớ','Hiểu','Áp dụng','Phân tích','Đánh giá','Sáng tạo']
print('OK')

In [ ]:
# Nạp dữ liệu và mô hình từ notebook 2.0
CLO_CSV = ROOT / 'data' / 'processed' / 'clos_labeled.csv'
if CLO_CSV.exists():
    df = pd.read_csv(CLO_CSV)
else:
    SYNTHETIC_CLOS = [
        ('841020', 'Trình bày định nghĩa biến và kiểu dữ liệu', 1, 0.85),
        ('841020', 'Liệt kê các toán tử cơ bản trong Python', 1, 0.90),
        ('841020', 'Giải thích cách hoạt động của vòng lặp for', 2, 0.88),
        ('841020', 'Mô tả sự khác biệt giữa list và tuple', 2, 0.82),
        ('841020', 'Viết chương trình tính giai thừa bằng đệ quy', 3, 0.91),
        ('841020', 'Áp dụng hàm lambda để xử lý danh sách', 3, 0.79),
        ('841303', 'Nêu định nghĩa cấu trúc stack và queue', 1, 0.93),
        ('841303', 'Giải thích độ phức tạp thuật toán O(n log n)', 2, 0.86),
        ('841303', 'Cài đặt cây nhị phân tìm kiếm bằng Java', 3, 0.88),
        ('841303', 'Phân tích ưu nhược điểm của các thuật toán sắp xếp', 4, 0.91),
        ('841303', 'Lựa chọn cấu trúc dữ liệu phù hợp cho bài toán cụ thể', 5, 0.75),
        ('841109', 'Kể tên các thành phần cơ bản của mạng máy tính', 1, 0.95),
        ('841109', 'Giải thích mô hình OSI 7 tầng', 2, 0.87),
        ('841109', 'Cấu hình router và switch cơ bản', 3, 0.82),
        ('841109', 'So sánh TCP và UDP về độ tin cậy và hiệu năng', 4, 0.88),
        ('841109', 'Thiết kế mạng LAN cho văn phòng 50 người', 6, 0.77),
        ('841434', 'Định nghĩa khái niệm transaction trong CSDL', 1, 0.92),
        ('841434', 'Mô tả mô hình quan hệ và các ràng buộc toàn vẹn', 2, 0.84),
        ('841434', 'Viết câu truy vấn SQL JOIN phức tạp', 3, 0.89),
        ('841434', 'Phân tích và chuẩn hoá lược đồ CSDL đến 3NF', 4, 0.85),
        ('841434', 'Đánh giá hiệu năng truy vấn và đề xuất tối ưu index', 5, 0.78),
        ('841470', 'Trình bày nguyên lý hoạt động của mạng nơ-ron', 1, 0.90),
        ('841470', 'Giải thích thuật toán gradient descent', 2, 0.86),
        ('841470', 'Huấn luyện mô hình phân loại đơn giản bằng scikit-learn', 3, 0.88),
        ('841470', 'Phân tích kết quả confusion matrix và đề xuất cải tiến', 4, 0.83),
        ('841470', 'Đánh giá và lựa chọn mô hình ML phù hợp cho bài toán thực', 5, 0.76),
        ('841470', 'Thiết kế pipeline ML end-to-end cho bài toán phân loại văn bản', 6, 0.72),
        ('841020', 'Có ý thức bảo vệ quyền sở hữu trí tuệ phần mềm', 0, 0.68),
    ]
    df = pd.DataFrame(SYNTHETIC_CLOS, columns=['course_code', 'text', 'bloom_level', 'ocr_confidence'])

df['bloom_level'] = df['bloom_level'].astype(int)
if 'course_code' not in df.columns:
    df['course_code'] = 'unknown'
if 'ocr_confidence' not in df.columns:
    df['ocr_confidence'] = 0.85

X = df['text'].values
y = df['bloom_level'].values
groups = df['course_code'].values
print(f'Dữ liệu: {len(df)} CLO')

## 1. Confusion Matrix 7×7 (Cross-val predictions)

In [ ]:
# Build features
tfidf_word = TfidfVectorizer(ngram_range=(1, 2), max_features=5000, analyzer='word')
tfidf_char = TfidfVectorizer(ngram_range=(2, 4), max_features=3000, analyzer='char_wb')
tfidf_word.fit(X); tfidf_char.fit(X)
X_feat = sp.hstack([tfidf_word.transform(X), tfidf_char.transform(X)])

cv = StratifiedGroupKFold(n_splits=min(5, len(np.unique(groups))), shuffle=True, random_state=RANDOM_SEED)
clf = LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000, random_state=RANDOM_SEED)

# Cross-val predictions để đánh giá công bằng
y_pred_cv = cross_val_predict(clf, X_feat, y, groups=groups, cv=cv)

# Confusion matrix
plot_confusion_matrix(
    y, y_pred_cv,
    labels=[f'L{i}' for i in range(7)],
    title='Confusion Matrix M2 Bloom (7×7, CV predictions)',
    save_path=str(FIG_DIR / 'm2_confusion_matrix.png'),
    normalize=True,
)
print('Đã lưu: reports/figures/m2_confusion_matrix.png')

print()
print(classification_report(y, y_pred_cv,
                             target_names=[f'L{i}: {n}' for i, n in enumerate(BLOOM_NAMES)],
                             zero_division=0))

## 2. Per-class F1 và Accuracy±1

In [ ]:
# Accuracy±1: tỷ lệ dự đoán trong khoảng ±1 lớp so với nhãn thực
acc_pm1 = np.mean(np.abs(y - y_pred_cv) <= 1)
exact_acc = accuracy_score(y, y_pred_cv)
macro_f1 = f1_score(y, y_pred_cv, average='macro', zero_division=0)

# Per-class F1
per_class_f1 = f1_score(y, y_pred_cv, average=None, zero_division=0, labels=list(range(7)))

print(f'Exact accuracy  : {exact_acc:.4f}')
print(f'Accuracy ±1 lớp : {acc_pm1:.4f}')
print(f'Macro-F1        : {macro_f1:.4f}')
print()
print('Per-class F1:')
for i, (name, f1) in enumerate(zip(BLOOM_NAMES, per_class_f1)):
    n_samples = (y == i).sum()
    print(f'  L{i} {name:<15}: F1={f1:.4f} (n={n_samples})')

## 3. Gộp 3 nhóm mức Bloom

In [ ]:
# Gộp lớp: {0,1,2} → 'Thấp', {3,4} → 'Trung', {5,6} → 'Cao'
def group_bloom(level):
    if level in (0, 1, 2): return 0  # Thấp
    elif level in (3, 4):  return 1  # Trung
    else:                  return 2  # Cao

y_grp = np.array([group_bloom(l) for l in y])
y_pred_grp = np.array([group_bloom(l) for l in y_pred_cv])
group_names = ['Thấp (0-2)', 'Trung (3-4)', 'Cao (5-6)']

grp_macro_f1 = f1_score(y_grp, y_pred_grp, average='macro', zero_division=0)
grp_acc = accuracy_score(y_grp, y_pred_grp)

print(f'Accuracy nhóm  : {grp_acc:.4f}')
print(f'Macro-F1 nhóm  : {grp_macro_f1:.4f}')
print()
print(classification_report(y_grp, y_pred_grp, target_names=group_names, zero_division=0))

# Confusion matrix nhóm
plot_confusion_matrix(
    y_grp, y_pred_grp,
    labels=group_names,
    title='Confusion Matrix M2 Bloom – 3 Nhóm',
    save_path=str(FIG_DIR / 'm2_confusion_grouped.png'),
    normalize=True,
)
print('Đã lưu: reports/figures/m2_confusion_grouped.png')

## 4. Phân tích 20 câu sai phân loại

In [ ]:
# Lấy các câu sai
errors_mask = (y != y_pred_cv)
error_df = df[errors_mask].copy()
error_df['pred'] = y_pred_cv[errors_mask]
error_df['true'] = y[errors_mask]
error_df['diff'] = abs(error_df['true'] - error_df['pred'])
error_df = error_df.sort_values('diff', ascending=False)

print(f'Tổng số câu sai: {len(error_df)} / {len(df)} ({len(error_df)/len(df)*100:.1f}%)')
print()

# Hiển thị tối đa 20 câu sai
n_show = min(20, len(error_df))
print(f'=== TOP {n_show} CÂU SAI (sắp xếp theo |true-pred|) ===')
for _, row in error_df.head(n_show).iterrows():
    print(f'  True=L{row["true"]}({BLOOM_NAMES[row["true"]]}), Pred=L{row["pred"]}({BLOOM_NAMES[row["pred"]]})')
    print(f'  Text: {str(row["text"])[:100]}')
    # Phân tích nguyên nhân
    diff = row['diff']
    if diff >= 3:
        reason = 'Lỗi nghiêm trọng: động từ mơ hồ hoặc OCR lỗi'
    elif diff == 2:
        reason = 'Lỗi trung bình: CLO kết hợp nhiều động từ'
    else:
        reason = 'Lỗi nhẹ: ranh giới lớp liền kề'
    print(f'  Nguyên nhân: {reason}')
    print()

## 5. Ảnh hưởng OCR Quality lên kết quả

In [ ]:
# Phân tích theo ngưỡng confidence
THRESHOLD = 0.85
high_conf = df['ocr_confidence'] >= THRESHOLD
low_conf = ~high_conf

results_conf = []
for label, mask in [('High conf (≥0.85)', high_conf), ('Low conf (<0.85)', low_conf)]:
    if mask.sum() > 0:
        y_t = y[mask.values]
        y_p = y_pred_cv[mask.values]
        f1 = f1_score(y_t, y_p, average='macro', zero_division=0)
        acc = accuracy_score(y_t, y_p)
        results_conf.append({'Nhóm': label, 'n': mask.sum(), 'Accuracy': acc, 'Macro-F1': f1})

print('=== Ảnh hưởng OCR Quality ===')
print(pd.DataFrame(results_conf).to_string(index=False))

# Bar chart
if results_conf:
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    for ax, metric in zip(axes, ['Accuracy', 'Macro-F1']):
        groups_label = [r['Nhóm'] for r in results_conf]
        vals = [r[metric] for r in results_conf]
        ax.bar(groups_label, vals, color=['#1565C0', '#F57F17'])
        ax.set_title(f'{metric} theo OCR Quality', fontweight='bold')
        ax.set_ylim(0, 1.1)
        ax.grid(axis='y', alpha=0.4)
        for i, v in enumerate(vals):
            ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'm2_ocr_impact.png', dpi=150, bbox_inches='tight')
    plt.close()
    print('Đã lưu: reports/figures/m2_ocr_impact.png')